# 代码文件2：Task1指标数据分析

本notebook用于分析task1的指标数据，创建task1指标数据表单和论文需要的图片。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# 表格美化显示
try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False
    print("提示: 安装 tabulate 可以获得更好的表格显示效果: pip install tabulate")

try:
    from rich.console import Console
    from rich.table import Table
    from rich import box
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    print("提示: 安装 rich 可以获得彩色表格显示效果: pip install rich")

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置图表美化样式
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'seaborn-darkgrid')
sns.set_palette("husl")  # 使用更鲜艳的颜色调色板
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# 优化pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)  # 不限制列宽，让内容完整显示
pd.set_option('display.precision', 3)
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if pd.notna(x) else '')
pd.set_option('display.expand_frame_repr', False)  # 不换行显示DataFrame

# 定义美化表格显示函数
def display_table(df, title="", max_rows=20, use_rich=True):
    """美化显示DataFrame表格"""
    if title:
        print(f"\n{'='*80}")
        print(f"{title:^80}")
        print(f"{'='*80}")
    
    # 优先使用rich显示（如果可用且数据不太大）
    if HAS_RICH and use_rich and len(df) <= 50 and len(df.columns) <= 15:
        console = Console(width=None)  # 不限制控制台宽度
        table = Table(show_header=True, header_style="bold magenta", box=box.ROUNDED, expand=True)
        
        # 添加列（不换行，允许更宽的列）
        for col in df.columns:
            table.add_column(str(col), style="cyan", no_wrap=True)
        
        # 添加行
        for idx, row in df.head(max_rows).iterrows():
            table.add_row(*[str(val) if pd.notna(val) else 'N/A' for val in row])
        
        console.print(table)
        if len(df) > max_rows:
            print(f"\n... (还有 {len(df) - max_rows} 行未显示)")
    
    # 使用tabulate显示
    elif HAS_TABULATE:
        if len(df) > max_rows:
            print(f"\n显示前{max_rows}行（共{len(df)}行）:")
            print(tabulate(df.head(max_rows), headers='keys', 
                          showindex=False, floatfmt='.3f', numalign='right'))
            print(f"\n... (还有 {len(df) - max_rows} 行)")
        else:
            print(tabulate(df, headers='keys', 
                          showindex=False, floatfmt='.3f', numalign='right'))
    
    # 回退到pandas默认显示
    else:
        if len(df) > max_rows:
            print(f"\n显示前{max_rows}行（共{len(df)}行）:")
            print(df.head(max_rows).to_string(max_colwidth=None))
            print(f"\n... (还有 {len(df) - max_rows} 行)")
        else:
            print(df.to_string(max_colwidth=None))
    
    print(f"\n形状: {df.shape[0]} 行 × {df.shape[1]} 列")

# 数据根目录
# Resolve VCG-Bench root from the current notebook working directory.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'src').exists():
            return path
    return start

BASE_DIR = find_repo_root()
DATA_DIR = BASE_DIR / 'data'
ANALYSIS_DIR = DATA_DIR / 'analysis'

# 创建必要的目录结构
(ANALYSIS_DIR / 'task1_analysis' / 'tables').mkdir(parents=True, exist_ok=True)
(ANALYSIS_DIR / 'task1_analysis' / 'figures').mkdir(parents=True, exist_ok=True)

print(f"数据根目录: {BASE_DIR}")
print(f"分析结果目录: {ANALYSIS_DIR}")


In [ ]:
# 读取Task1详细结果和难度分层数据

# 读取detailed_results.json
task1_results_path = DATA_DIR / 'task1_evaluation' / 'detailed_results.json'
with open(task1_results_path, 'r', encoding='utf-8') as f:
    task1_data = json.load(f)

# 检查JSON结构：可能是字典（包含samples键）或直接是列表
if isinstance(task1_data, dict) and 'samples' in task1_data:
    task1_results = task1_data['samples']
elif isinstance(task1_data, list):
    task1_results = task1_data
else:
    task1_results = []

print(f"Task1结果记录数: {len(task1_results)}")

# 读取难度分层数据
difficulty_levels_path = ANALYSIS_DIR / 'difficulty_stratification' / 'tables' / 'image_difficulty_levels.csv'
difficulty_levels_df = pd.read_csv(difficulty_levels_path)

print(f"难度分层数据记录数: {len(difficulty_levels_df)}")
display_table(difficulty_levels_df.head(5), title="难度分层数据前5条", max_rows=5)


In [ ]:
# 数据清洗和合并
# 将JSON数据转换为DataFrame，并合并难度信息

def extract_metric_score(metric_data):
    """提取指标的score值，如果success=false或值为None则返回None"""
    if metric_data is None:
        return None
    if isinstance(metric_data, dict):
        if metric_data.get('success', True) == False:
            return None
        return metric_data.get('score', None)
    return metric_data

# 展开JSON数据为DataFrame
rows = []
for item in task1_results:
    row = {
        'sample_id': item.get('sample_id'),
        'domain': item.get('domain'),
        'model': item.get('model'),
    }
    
    # 提取各指标的score值
    metrics = item.get('metrics', {})
    row['style_consistency_score'] = extract_metric_score(metrics.get('style_consistency_score'))
    # 从JSON读取codevqa，但在DataFrame中列名改为codexqa
    row['codexqa'] = extract_metric_score(metrics.get('codevqa'))
    row['execution_success_rate'] = extract_metric_score(metrics.get('execution_success_rate'))
    row['xml_token_count'] = extract_metric_score(metrics.get('xml_token_count'))
    row['siglip_score'] = extract_metric_score(metrics.get('siglip_score'))
    
    rows.append(row)

task1_df = pd.DataFrame(rows)

# 清洗：删除xml_token_count为0的样本
task1_df = task1_df[task1_df['xml_token_count'] != 0].copy()

# 合并难度信息
task1_df = task1_df.merge(
    difficulty_levels_df[['sample_id', 'domain', 'difficulty_level']],
    on=['sample_id', 'domain'],
    how='left'
)

# 保存合并后的数据
output_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_merged_data.csv'
task1_df.to_csv(output_path, index=False, encoding='utf-8')

print(f"合并后数据记录数: {len(task1_df)}")
print(f"列数: {len(task1_df.columns)}")
print(f"\n列名: {list(task1_df.columns)}")

# 显示缺失值统计
missing_stats = pd.DataFrame({
    '缺失数量': task1_df.isnull().sum(),
    '缺失比例(%)': (task1_df.isnull().sum() / len(task1_df) * 100).round(2)
})
missing_stats = missing_stats[missing_stats['缺失数量'] > 0].sort_values('缺失数量', ascending=False)
if len(missing_stats) > 0:
    display_table(missing_stats, title="缺失值统计")
else:
    print("\n✓ 无缺失值")

# 显示前5条数据
display_table(task1_df.head(5), title="前5条数据预览")
print(f"\n已保存到: {output_path}")


In [ ]:
# 按模型分组统计
# 对每个模型的指标计算统计值（排除None/Null/success=false的样本）

def safe_statistics(series, stat_name):
    """安全计算统计量，排除None/Null值"""
    valid_series = series.dropna()
    if len(valid_series) == 0:
        return None
    
    if stat_name == 'mean':
        return valid_series.mean()
    elif stat_name == 'median':
        return valid_series.median()
    elif stat_name == 'std':
        return valid_series.std()
    elif stat_name == 'min':
        return valid_series.min()
    elif stat_name == 'max':
        return valid_series.max()
    elif stat_name == 'count':
        return len(valid_series)
    return None

# 定义要统计的指标
metrics_list = ['style_consistency_score', 'codexqa', 'execution_success_rate', 
                'xml_token_count', 'siglip_score']
statistics_list = ['mean', 'median', 'std', 'min', 'max', 'count']

# 按模型分组统计
model_stats = []
for model in task1_df['model'].unique():
    model_data = task1_df[task1_df['model'] == model]
    
    row = {'model': model}
    
    for metric in metrics_list:
        metric_series = model_data[metric]
        for stat in statistics_list:
            col_name = f'{metric}_{stat}'
            row[col_name] = safe_statistics(metric_series, stat)
    
    row['valid_samples_count'] = len(model_data)
    model_stats.append(row)

task1_by_model_df = pd.DataFrame(model_stats)

# 保存统计表格
output_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_by_model.csv'
task1_by_model_df.to_csv(output_path, index=False, encoding='utf-8')

# 显示所有数据（所有模型和所有均值列）
display_cols = ['model'] + [col for col in task1_by_model_df.columns if '_mean' in col]
# 确保至少显示前11行（如果数据少于11行则显示全部）
min_display_rows = min(11, len(task1_by_model_df))
display_table(task1_by_model_df[display_cols], title="Task1按模型分组统计（主要指标均值）", max_rows=max(min_display_rows, len(task1_by_model_df)))
print(f"\n已保存到: {output_path}")

# 显示完整表格（display_cols，保留3位小数，列名完整显示）
# 确保所有数值列保留3位小数
task1_by_model_display = task1_by_model_df[display_cols].copy()
for col in task1_by_model_display.columns:
    if task1_by_model_display[col].dtype in ['float64', 'float32']:
        task1_by_model_display[col] = task1_by_model_display[col].round(3)

display_table(task1_by_model_display, title="Task1按模型分组统计（主要指标均值 - 3位小数版本）", max_rows=len(task1_by_model_display))


In [ ]:
# 绘制模型对比图（图1）
# 使用热力图展示不同模型的性能对比

# 选择主要指标
main_metrics = ['style_consistency_score_mean', 'codexqa_mean', 
                'execution_success_rate_mean', 'siglip_score_mean']

# 构建矩阵：行=模型，列=指标
heatmap_data = task1_by_model_df.set_index('model')[main_metrics].copy()

# 将数据转换为数值类型，非数值（如None、字符串）转换为NaN
for col in heatmap_data.columns:
    heatmap_data[col] = pd.to_numeric(heatmap_data[col], errors='coerce')

# 重命名列以便显示
heatmap_data.columns = ['SCS', 'CodeXQA', 'Exec Success', 'SigLIP']

# 绘制热力图
plt.figure(figsize=(12, max(8, len(heatmap_data) * 0.5)))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            cbar_kws={'label': 'Score'}, linewidths=0.5, linecolor='gray',
            mask=heatmap_data.isna())  # 将NaN值用mask隐藏
plt.title('Task1 Model Performance Comparison (Heatmap)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Metrics', fontsize=12)
plt.ylabel('Models', fontsize=12)
plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_model_comparison.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 按难度等级分组统计
# 计算每个模型在不同难度等级下的表现

difficulty_stats = []
for difficulty in task1_df['difficulty_level'].dropna().unique():
    for model in task1_df['model'].unique():
        model_difficulty_data = task1_df[
            (task1_df['model'] == model) & 
            (task1_df['difficulty_level'] == difficulty)
        ]
        
        if len(model_difficulty_data) == 0:
            continue
        
        row = {
            'difficulty_level': difficulty,
            'model': model
        }
        
        for metric in metrics_list:
            metric_series = model_difficulty_data[metric]
            for stat in statistics_list:
                col_name = f'{metric}_{stat}'
                row[col_name] = safe_statistics(metric_series, stat)
        
        row['valid_samples_count'] = len(model_difficulty_data)
        difficulty_stats.append(row)

task1_by_difficulty_df = pd.DataFrame(difficulty_stats)

# 保存统计表格
output_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_by_difficulty.csv'
task1_by_difficulty_df.to_csv(output_path, index=False, encoding='utf-8')

# 选择主要列进行显示
display_cols = ['difficulty_level', 'model'] + [col for col in task1_by_difficulty_df.columns if '_mean' in col]
display_table(task1_by_difficulty_df[display_cols], title="Task1按难度等级分组统计（主要指标）")
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制按难度等级的性能表现图（图2）
# 使用分组柱状图展示模型在不同难度下的表现

# 选择主要指标和模型
main_metrics_plot = ['style_consistency_score_mean', 'codexqa_mean']
# 使用排序后的模型列表，确保包含所有重要模型（特别是Qwen 32B）
all_models = sorted(task1_by_difficulty_df['model'].unique())
top_models = all_models[:11]  # 选择前11个模型（按字母排序）
print(f"选择的模型: {top_models}")

fig, axes = plt.subplots(1, len(main_metrics_plot), figsize=(16, 6))

for idx, metric in enumerate(main_metrics_plot):
    ax = axes[idx]
    
        # 准备数据
    plot_data = task1_by_difficulty_df[
        task1_by_difficulty_df['model'].isin(top_models)
    ].pivot_table(
        index='difficulty_level',
        columns='model',
        values=metric
    )
    
    # 按难度等级排序：Easy -> Medium -> Hard
    difficulty_order = ['Easy', 'Medium', 'Hard']
    # 只保留存在的难度等级
    available_difficulties = [d for d in difficulty_order if d in plot_data.index]
    plot_data = plot_data.reindex(available_difficulties)
    
    # 绘制分组柱状图
    plot_data.plot(kind='bar', ax=ax, width=0.8, alpha=0.8)
    
    # 设置标题，将codexqa显示为CodeXQA
    if 'codexqa' in metric.lower():
        metric_name = 'CodeXQA'
    else:
        metric_name = metric.replace('_mean', '').replace('_', ' ').title()
    ax.set_title(f'{metric_name} by Difficulty Level', fontsize=12, fontweight='bold')
    ax.set_xlabel('Difficulty Level', fontsize=11)
    ax.set_ylabel('Score', fontsize=11)
    ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_by_difficulty.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 绘制按模型和难度等级的性能表现图（图2b）
# x轴是每个模型，显示每个模型在不同难度级别下的两个指标表现

# 选择主要指标和模型
main_metrics_plot = ['style_consistency_score_mean', 'codexqa_mean']
metric_labels = ['Style Consistency Score', 'CodeXQA']
# 使用排序后的模型列表
all_models = sorted(task1_by_difficulty_df['model'].unique())
top_models = all_models[:11]  # 选择前11个模型（按字母排序）
print(f"选择的模型: {top_models}")

# 按难度等级排序
difficulty_order = ['Easy', 'Medium', 'Hard']
num_models = len(top_models)
num_difficulties = len(difficulty_order)

# 创建子图：两个指标分别显示
fig, axes = plt.subplots(1, len(main_metrics_plot), figsize=(18, 6))

for idx, (metric, metric_label) in enumerate(zip(main_metrics_plot, metric_labels)):
    ax = axes[idx]
    
    # 计算合适的bar宽度：确保每组bar不会重叠
    max_group_width = 0.85  # 每组bar的最大总宽度
    width = max_group_width / num_difficulties  # 根据难度等级数量动态调整宽度
    
    # 准备数据
    x = np.arange(num_models)
    
    # 为每个难度等级绘制bar
    colors = plt.cm.Pastel1(np.linspace(0, 1, num_difficulties))  # 使用不同的颜色
    for i, difficulty in enumerate(difficulty_order):
        scores = []
        for model in top_models:
            model_difficulty_data = task1_by_difficulty_df[
                (task1_by_difficulty_df['model'] == model) & 
                (task1_by_difficulty_df['difficulty_level'] == difficulty)
            ]
            score_val = model_difficulty_data[metric].values[0] if len(model_difficulty_data) > 0 else None
            # 处理None值，转换为0或NaN
            if score_val is None or pd.isna(score_val):
                score = 0
            else:
                score = pd.to_numeric(score_val, errors='coerce')
                if pd.isna(score):
                    score = 0
            scores.append(score)
        
        # 计算每组内的偏移量，使整个组居中
        offset_in_group = (i - (num_difficulties - 1) / 2) * width
        ax.bar(x + offset_in_group, scores, width, label=difficulty, alpha=0.8, color=colors[i])
    
    # 设置x轴标签（模型名称的简短形式）
    model_labels = [model.split('/')[-1] if '/' in model else model for model in top_models]
    ax.set_xlabel('Model', fontsize=12)
    
    # 设置y轴标签和范围
    if 'codexqa' in metric.lower():
        ax.set_ylabel('Accuracy', fontsize=12)
        ax.set_ylim([0, 1])
    else:
        ax.set_ylabel('Score', fontsize=12)
        ax.set_ylim([0, 10])
    
    ax.set_title(f'{metric_label} by Model and Difficulty Level', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=9)
    ax.legend(title='Difficulty Level', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    # 设置x轴范围，确保有足够的空间显示所有bar
    ax.set_xlim([-0.5, num_models - 0.5])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_by_model_and_difficulty.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()



In [ ]:
# 读取并处理CodeXQA按问题类型数据
# 注意：原始文件名是codevqa，但我们在代码中使用codexqa作为变量名和显示名

codevqa_path = DATA_DIR / 'task1_evaluation' / 'all_models_codevqa_by_question_type.csv'
task1_codexqa_df = pd.read_csv(codevqa_path)

# 数据验证和清洗
print(f"CodeXQA数据记录数: {len(task1_codexqa_df)}")
display_table(task1_codexqa_df.head(10), title="CodeXQA数据概览")

# 显示问题类型分布
qtype_dist = pd.DataFrame({
    '数量': task1_codexqa_df['question_type'].value_counts(),
    '占比(%)': (task1_codexqa_df['question_type'].value_counts(normalize=True) * 100).round(2)
})
display_table(qtype_dist, title="问题类型分布")

# 保存（如果需要处理）
output_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_codexqa_by_question_type.csv'
task1_codexqa_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制CodeXQA按问题类型表现图（图3）
# 展示模型在不同问题类型上的表现

# 选择主要模型（使用排序后的列表，确保包含所有重要模型）
all_models_cxqa = sorted(task1_codexqa_df['model'].unique())
top_models_cxqa = all_models_cxqa[:11]  # 选择前11个模型（按字母排序）
print(f"选择的模型: {top_models_cxqa}")

fig, ax = plt.subplots(figsize=(14, 6))

# 定义问题类型
question_types = ['counting', 'identification', 'relationship']
num_types = len(question_types)
num_models = len(top_models_cxqa)

# 计算合适的bar宽度：确保每组bar不会重叠
# 每个问题类型之间间距为1，每组bar总宽度应该小于0.9，留出间距
max_group_width = 0.85  # 每组bar的最大总宽度
width = max_group_width / num_models  # 根据模型数量动态调整宽度

# 计算每组bar的起始位置，使整个组居中
x = np.arange(num_types)
group_spacing = 1.0  # 问题类型之间的间距

for i, model in enumerate(top_models_cxqa):
    model_data = task1_codexqa_df[task1_codexqa_df['model'] == model]
    accuracies = []
    for qtype in question_types:
        qtype_data = model_data[model_data['question_type'] == qtype]
        acc = qtype_data['accuracy'].values[0] if len(qtype_data) > 0 else 0
        accuracies.append(acc)
    
    model_short = model.split('/')[-1] if '/' in model else model
    # 计算每组内的偏移量，使整个组居中
    offset_in_group = (i - (num_models - 1) / 2) * width
    ax.bar(x + offset_in_group, accuracies, width, label=model_short, alpha=0.8)

ax.set_xlabel('Question Type', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('CodeXQA Performance by Question Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Counting', 'Identification', 'Relationship'])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
# 设置x轴范围，确保有足够的空间显示所有bar
ax.set_xlim([-0.5, num_types - 0.5])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_codexqa_by_question_type.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 绘制CodeXQA按模型表现图（图3b）
# x轴是每个模型，显示每个模型在不同问题类型上的表现

# 选择主要模型
all_models_cxqa = sorted(task1_codexqa_df['model'].unique())
top_models_cxqa = all_models_cxqa[:11]  # 选择前11个模型
print(f"选择的模型: {top_models_cxqa}")

fig, ax = plt.subplots(figsize=(16, 6))

# 定义问题类型
question_types = ['counting', 'identification', 'relationship']
question_type_labels = ['Counting', 'Identification', 'Relationship']
num_models = len(top_models_cxqa)
num_types = len(question_types)

# 计算合适的bar宽度：确保每组bar不会重叠
max_group_width = 0.85  # 每组bar的最大总宽度
width = max_group_width / num_types  # 根据问题类型数量动态调整宽度

# 准备数据
x = np.arange(num_models)

# 为每个问题类型绘制bar
colors = plt.cm.Set3(np.linspace(0, 1, num_types))  # 使用不同的颜色
for i, (qtype, qlabel) in enumerate(zip(question_types, question_type_labels)):
    accuracies = []
    for model in top_models_cxqa:
        model_data = task1_codexqa_df[task1_codexqa_df['model'] == model]
        qtype_data = model_data[model_data['question_type'] == qtype]
        acc = qtype_data['accuracy'].values[0] if len(qtype_data) > 0 else 0
        accuracies.append(acc)
    
    # 计算每组内的偏移量，使整个组居中
    offset_in_group = (i - (num_types - 1) / 2) * width
    ax.bar(x + offset_in_group, accuracies, width, label=qlabel, alpha=0.8, color=colors[i])

# 设置x轴标签（模型名称的简短形式）
model_labels = [model.split('/')[-1] if '/' in model else model for model in top_models_cxqa]
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('CodeXQA Performance by Model and Question Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=9)
ax.legend(title='Question Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
# 设置x轴范围，确保有足够的空间显示所有bar
ax.set_xlim([-0.5, num_models - 0.5])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_codexqa_by_model.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()

In [ ]:
# 读取并处理SCS按维度数据

scs_path = DATA_DIR / 'task1_evaluation' / 'all_models_scs_by_dimension.csv'
task1_scs_df = pd.read_csv(scs_path)

# 数据验证和清洗
print(f"SCS数据记录数: {len(task1_scs_df)}")
display_table(task1_scs_df.head(10), title="SCS数据概览")

# 显示维度分布
dim_dist = pd.DataFrame({
    '数量': task1_scs_df['dimension'].value_counts(),
    '占比(%)': (task1_scs_df['dimension'].value_counts(normalize=True) * 100).round(2)
})
display_table(dim_dist, title="SCS维度分布")

# 保存（如果需要处理）
output_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_scs_by_dimension.csv'
task1_scs_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制SCS按维度表现图（图4）
# 展示模型在不同SCS维度上的表现

# 选择主要模型（使用排序后的列表，确保包含所有重要模型）
all_models_scs = sorted(task1_scs_df['model'].unique())
print(f"SCS数据中实际有 {len(all_models_scs)} 个模型")
print(f"所有模型: {all_models_scs}")

# 显示所有可用的模型（不限制数量）
top_models_scs = all_models_scs  # 显示所有可用模型
print(f"\n选择的模型（共{len(top_models_scs)}个）: {top_models_scs}")

fig, ax = plt.subplots(figsize=(14, 6))

# 定义SCS维度
dimensions = ['visual_style_consistency', 'layout_consistency', 'aesthetic_quality']
dimension_labels = ['Visual Style', 'Layout', 'Aesthetic']
num_dims = len(dimensions)
num_models = len(top_models_scs)

# 计算合适的bar宽度：确保每组bar不会重叠
# 每个维度之间间距为1，每组bar总宽度应该小于0.9，留出间距
max_group_width = 0.85  # 每组bar的最大总宽度
width = max_group_width / num_models  # 根据模型数量动态调整宽度

# 计算每组bar的起始位置，使整个组居中
x = np.arange(num_dims)

for i, model in enumerate(top_models_scs):
    model_data = task1_scs_df[task1_scs_df['model'] == model]
    scores = []
    stds = []
    for dim in dimensions:
        dim_data = model_data[model_data['dimension'] == dim]
        score = dim_data['mean_score'].values[0] if len(dim_data) > 0 else 0
        std = dim_data['std_score'].values[0] if len(dim_data) > 0 else 0
        scores.append(score)
        stds.append(std)
    
    model_short = model.split('/')[-1] if '/' in model else model
    # 计算每组内的偏移量，使整个组居中
    offset_in_group = (i - (num_models - 1) / 2) * width
    bars = ax.bar(x + offset_in_group, scores, width, label=model_short, alpha=0.8)
    # 添加误差棒
    ax.errorbar(x + offset_in_group, scores, yerr=stds, fmt='none', color='black', capsize=3, linewidth=1)

ax.set_xlabel('SCS Dimension', fontsize=12)
ax.set_ylabel('Mean Score', fontsize=12)
ax.set_title('Style Consistency Score by Dimension', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(dimension_labels, fontsize=10)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_ylim([0, 10])
ax.grid(True, alpha=0.3, axis='y')
# 设置x轴范围，确保有足够的空间显示所有bar
ax.set_xlim([-0.5, num_dims - 0.5])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_scs_by_dimension.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 绘制SCS按模型表现图（图4b）
# x轴是每个模型，显示每个模型在不同SCS维度上的表现

# 选择主要模型
all_models_scs = sorted(task1_scs_df['model'].unique())
top_models_scs = all_models_scs[:11]  # 选择前11个模型
print(f"选择的模型: {top_models_scs}")

fig, ax = plt.subplots(figsize=(16, 6))

# 定义SCS维度
dimensions = ['visual_style_consistency', 'layout_consistency', 'aesthetic_quality']
dimension_labels = ['Visual Style', 'Layout', 'Aesthetic']
num_models = len(top_models_scs)
num_dims = len(dimensions)

# 计算合适的bar宽度：确保每组bar不会重叠
max_group_width = 0.85  # 每组bar的最大总宽度
width = max_group_width / num_dims  # 根据维度数量动态调整宽度

# 准备数据
x = np.arange(num_models)

# 为每个维度绘制bar
colors = plt.cm.Set2(np.linspace(0, 1, num_dims))  # 使用不同的颜色
for i, (dim, dim_label) in enumerate(zip(dimensions, dimension_labels)):
    scores = []
    stds = []
    for model in top_models_scs:
        model_data = task1_scs_df[task1_scs_df['model'] == model]
        dim_data = model_data[model_data['dimension'] == dim]
        score = dim_data['mean_score'].values[0] if len(dim_data) > 0 else 0
        std = dim_data['std_score'].values[0] if len(dim_data) > 0 else 0
        scores.append(score)
        stds.append(std)
    
    # 计算每组内的偏移量，使整个组居中
    offset_in_group = (i - (num_dims - 1) / 2) * width
    bars = ax.bar(x + offset_in_group, scores, width, label=dim_label, alpha=0.8, color=colors[i])
    # 添加误差棒
    ax.errorbar(x + offset_in_group, scores, yerr=stds, fmt='none', color='black', capsize=3, linewidth=1)

# 设置x轴标签（模型名称的简短形式）
model_labels = [model.split('/')[-1] if '/' in model else model for model in top_models_scs]
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Mean Score', fontsize=12)
ax.set_title('Style Consistency Score by Model and Dimension', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=9)
ax.legend(title='SCS Dimension', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.set_ylim([0, 10])
ax.grid(True, alpha=0.3, axis='y')
# 设置x轴范围，确保有足够的空间显示所有bar
ax.set_xlim([-0.5, num_models - 0.5])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_scs_by_model.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 绘制SCS按难度等级表现折线图
# 展示每个模型在不同难度下的实际SCS分数表现

# 难度等级顺序
difficulty_order = ['Easy', 'Medium', 'Hard']

# 获取所有模型
all_models = sorted(task1_by_difficulty_df['model'].unique())

# 为每个模型准备数据
model_scores = {}
for model in all_models:
    scores = []
    for difficulty in difficulty_order:
        model_data = task1_by_difficulty_df[
            (task1_by_difficulty_df['model'] == model) &
            (task1_by_difficulty_df['difficulty_level'] == difficulty)
        ]
        if len(model_data) > 0:
            score = model_data['style_consistency_score_mean'].values[0]
            scores.append(score)
        else:
            scores.append(None)  # 如果没有数据，使用None
    model_scores[model] = scores

# 创建折线图
fig, ax = plt.subplots(figsize=(12, 7))

# 定义颜色和样式
colors = plt.cm.tab10(np.linspace(0, 1, len(all_models)))
markers = ['o', 's', '^', 'v', 'D', 'p', '*', 'h', 'X', 'd', '+']

# 为每个模型绘制一条线
for idx, model in enumerate(all_models):
    scores = model_scores[model]
    # 只绘制有数据的点
    valid_scores = [s for s in scores if s is not None]
    if len(valid_scores) > 0:
        # 获取模型简短名称
        model_short = model.split('/')[-1] if '/' in model else model
        ax.plot(difficulty_order, scores, 
                marker=markers[idx % len(markers)], 
                markersize=8, 
                linewidth=2, 
                label=model_short, 
                color=colors[idx],
                alpha=0.8)

# 设置图表属性
ax.set_xlabel('Difficulty Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Task 1 SCS Score', fontsize=14, fontweight='bold')
ax.set_title('SCS Score by Difficulty Level for Each Model', 
             fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

# 设置y轴范围
all_scores = [s for scores in model_scores.values() for s in scores if s is not None]
if all_scores:
    ax.set_ylim([0, max(all_scores) * 1.1])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_scs_by_difficulty_line.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

# 打印数据摘要
print("\n数据摘要:")
for model in all_models:
    scores = model_scores[model]
    model_short = model.split('/')[-1] if '/' in model else model
    easy_str = f"{scores[0]:.3f}" if scores[0] is not None else "N/A"
    medium_str = f"{scores[1]:.3f}" if scores[1] is not None else "N/A"
    hard_str = f"{scores[2]:.3f}" if scores[2] is not None else "N/A"
    print(f"{model_short}: Easy={easy_str}, Medium={medium_str}, Hard={hard_str}")

plt.show()


In [ ]:
# 绘制CodeXQA按难度等级表现折线图
# 展示每个模型在不同难度下的实际CodeXQA准确率表现

# 难度等级顺序
difficulty_order = ['Easy', 'Medium', 'Hard']

# 获取所有模型
all_models = sorted(task1_by_difficulty_df['model'].unique())

# 为每个模型准备数据
model_scores = {}
for model in all_models:
    scores = []
    for difficulty in difficulty_order:
        model_data = task1_by_difficulty_df[
            (task1_by_difficulty_df['model'] == model) &
            (task1_by_difficulty_df['difficulty_level'] == difficulty)
        ]
        if len(model_data) > 0:
            score_val = model_data['codexqa_mean'].values[0]
            # 处理None值
            if score_val is None or pd.isna(score_val):
                score = None
            else:
                score = pd.to_numeric(score_val, errors='coerce')
                if pd.isna(score):
                    score = None
            scores.append(score)
        else:
            scores.append(None)  # 如果没有数据，使用None
    model_scores[model] = scores

# 创建折线图
fig, ax = plt.subplots(figsize=(12, 7))

# 定义颜色和样式
colors = plt.cm.tab10(np.linspace(0, 1, len(all_models)))
markers = ['o', 's', '^', 'v', 'D', 'p', '*', 'h', 'X', 'd', '+']

# 为每个模型绘制一条线
for idx, model in enumerate(all_models):
    scores = model_scores[model]
    # 只绘制有数据的点
    valid_scores = [s for s in scores if s is not None]
    if len(valid_scores) > 0:
        # 获取模型简短名称
        model_short = model.split('/')[-1] if '/' in model else model
        ax.plot(difficulty_order, scores, 
                marker=markers[idx % len(markers)], 
                markersize=8, 
                linewidth=2, 
                label=model_short, 
                color=colors[idx],
                alpha=0.8)

# 设置图表属性
ax.set_xlabel('Difficulty Level', fontsize=14, fontweight='bold')
ax.set_ylabel('CodeXQA Accuracy', fontsize=14, fontweight='bold')
ax.set_title('CodeXQA Accuracy by Difficulty Level for Each Model', 
             fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

# 设置y轴范围（CodeXQA是0-1的准确率）
ax.set_ylim([0, 1.05])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task1_analysis' / 'figures' / 'task1_codexqa_by_difficulty_line.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

# 打印数据摘要
print("\n数据摘要:")
for model in all_models:
    scores = model_scores[model]
    model_short = model.split('/')[-1] if '/' in model else model
    easy_str = f"{scores[0]:.3f}" if scores[0] is not None else "N/A"
    medium_str = f"{scores[1]:.3f}" if scores[1] is not None else "N/A"
    hard_str = f"{scores[2]:.3f}" if scores[2] is not None else "N/A"
    print(f"{model_short}: Easy={easy_str}, Medium={medium_str}, Hard={hard_str}")

plt.show()


In [ ]:
# 指标说明文档

print("="*80)
print("Task1指标说明")
print("="*80)

print("\n1. Style Consistency Score (SCS) - 风格一致性分数")
print("   评分范围: 0-10分")
print("   评分维度:")
print("     - visual_style_consistency (视觉风格一致性): 评估颜色、字体、线条样式等视觉元素的一致性")
print("     - layout_consistency (布局一致性): 评估元素排列、对齐、间距等布局特征的一致性")
print("     - aesthetic_quality (美学质量): 评估整体视觉效果的美观程度")

print("\n2. CodeXQA - 代码视觉问答准确率")
print("   评分范围: 0-1（准确率）")
print("   问题类型:")
print("     - counting (计数类): 统计图表中特定元素的数量")
print("     - identification (识别类): 识别图表中特定元素的内容或属性")
print("     - relationship (关系类): 理解图表中元素之间的关系")

print("\n3. Execution Success Rate - XML执行成功率")
print("   评分范围: 0-1（成功率）")
print("   说明: 评估生成的XML代码能否成功渲染")

print("\n4. XML Token Count - XML token数")
print("   说明: 生成的XML代码的token数量，反映代码复杂度")


print("\n5. SigLIP Score - SigLIP相似度分数")
print("   评分范围: 0-1（相似度）")
print("   说明: 使用SigLIP模型评估生成图像与原始图像的相似度")
print("="*80)
